# LeetCode 3299 — Sum of Consecutive Subsequences bằng Python

- Link: https://leetcode.com/problems/sum-of-consecutive-subsequences/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **DP theo giá trị kết thúc** mang cặp `(số dãy, tổng)`, tách **hai chiều** (tăng 1 / giảm 1) và trừ phần **đếm trùng** (dãy một phần tử); cách nhìn **đóng góp trái × phải**

> **Ghi chú Premium:** bài Premium nên LeetCode không trả đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `getSum(self, nums)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3299/) — vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton (theo doocs):

```python
class Solution:
    def getSum(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Hiểu định nghĩa "liên tiếp": **toàn bộ** các bước cùng là `+1` (tăng) **hoặc** cùng là `−1` (giảm) — khác bài 3351 (cho phép đổi chiều).
2. Tách thành hai bài con độc lập: dãy con tăng-1 và dãy con giảm-1; dãy một phần tử thuộc **cả hai** ⇒ trừ một lần `Σ nums`.
3. Với mỗi chiều: DP `cnt[v]`, `sm[v]` theo giá trị kết thúc; nối `x` sau các dãy kết thúc bằng `x − 1` (tăng) hoặc `x + 1` (giảm).
4. Cách nhìn đóng góp: phần tử `i` thuộc `(1 + L_i)(1 + R_i)` dãy tăng-1, với `L_i` = số dãy tăng-1 kết thúc ngay trước (giá trị `x − 1`), `R_i` = số dãy bắt đầu ngay sau (giá trị `x + 1`).
5. Kiểm chứng bằng vét cạn.

## 2. Đề bài & ví dụ

Mảng `arr` độ dài `n` gọi là **liên tiếp** nếu `arr[i] − arr[i−1] = 1` với mọi `i`, **hoặc** `arr[i] − arr[i−1] = −1` với mọi `i` (mảng một phần tử cũng liên tiếp). **Giá trị** của mảng là tổng các phần tử. Cho `nums`, trả về tổng giá trị của **mọi dãy con khác rỗng liên tiếp** (theo vị trí), modulo `10^9 + 7`.

**Ví dụ 1:** `[1,2]` → `6` (`[1], [2], [1,2]`).
**Ví dụ 2:** `[1,4,2,3]` → `31` (`[1],[4],[2],[3],[1,2],[2,3],[4,3],[1,2,3]`).

**Ràng buộc (theo doocs):** `1 ≤ n ≤ 10^5`, `1 ≤ nums[i] ≤ 10^5`.

Ví dụ phản: `[3, 4, 3]` **không** liên tiếp (đổi chiều), `[8, 6]` không liên tiếp.

In [1]:
from typing import List
from collections import defaultdict
MOD = 10 ** 9 + 7

examples = [([1, 2], 6), ([1, 4, 2, 3], 31)]
for nums, exp in examples:
    print(nums, '-> kỳ vọng', exp)

[1, 2] -> kỳ vọng 6
[1, 4, 2, 3] -> kỳ vọng 31


## 3. Kiến thức nền

### 3.1. DP mang `(số lượng, tổng)`

`c` dãy có tổng các tổng `S`; nối `x` vào mỗi dãy ⇒ `c` dãy mới, tổng `S + x · c`.

### 3.2. Hai chiều và phần đếm trùng

Gọi `INC` = tổng giá trị mọi dãy con tăng-1 (kể cả độ dài 1), `DEC` tương tự cho giảm-1. Dãy độ dài `≥ 2` thuộc đúng một trong hai; dãy độ dài 1 thuộc cả hai ⇒ đáp án `= INC + DEC − Σ nums`.

In [2]:
def one_direction(nums, d):
    """Tổng giá trị mọi dãy con có bước luôn bằng d (d = +1 hoặc -1), kể cả độ dài 1."""
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        c = 1 + cnt[x - d]
        s = x * c + sm[x - d]
        cnt[x] += c; sm[x] += s
    return sum(sm.values())

print('INC =', one_direction([1, 4, 2, 3], 1), '| DEC =', one_direction([1, 4, 2, 3], -1), '| Σ nums =', 10)

INC = 24 | DEC = 17 | Σ nums = 10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 `[1, 4, 2, 3]`:

- Tăng-1: `[1],[4],[2],[3],[1,2],[2,3],[1,2,3]` ⇒ `1+4+2+3+3+5+6 = 24`.
- Giảm-1: `[1],[4],[2],[3],[4,3]` ⇒ `10 + 7 = 17`.
- Đáp án `24 + 17 − 10 = 31` ✓.

## 5. Cách 1 — Brute Force: mọi dãy con

In [3]:
def is_consecutive(seq):
    if len(seq) <= 1:
        return True
    d = seq[1] - seq[0]
    return d in (1, -1) and all(b - a == d for a, b in zip(seq, seq[1:]))

def brute(nums):
    n, total = len(nums), 0
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if is_consecutive(seq):
            total += sum(seq)
    return total % MOD

for nums, exp in examples:
    print(brute(nums), '| kỳ vọng', exp)

6 | kỳ vọng 6
31 | kỳ vọng 31


**Độ phức tạp:** `O(2^n · n)`.

## 6. Ý tưởng thuật toán / bước đột phá

```
INC: c = 1 + cntI[x-1];  s = x·c + smI[x-1];  cntI[x] += c;  smI[x] += s
DEC: c = 1 + cntD[x+1];  s = x·c + smD[x+1];  cntD[x] += c;  smD[x] += s
ans = Σ smI + Σ smD − Σ nums   (mod p)
```

Một lần duyệt, `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo chỉ số `O(n²)`

In [4]:
def dp_quadratic(nums):
    n = len(nums)
    total = 0
    for d in (1, -1):
        c, s = [0] * n, [0] * n
        for i, x in enumerate(nums):
            c[i], s[i] = 1, x
            for j in range(i):
                if x - nums[j] == d:
                    c[i] += c[j]; s[i] += s[j] + x * c[j]
            c[i] %= MOD; s[i] %= MOD
        total += sum(s)
    return (total - sum(nums)) % MOD

for nums, exp in examples:
    print(dp_quadratic(nums), '| kỳ vọng', exp)

6 | kỳ vọng 6
31 | kỳ vọng 31


### 7.2. Cách 3 — DP theo giá trị, hai chiều trong một lần duyệt (lời giải chính)

In [5]:
def dp_value(nums):
    cI, sI, cD, sD = defaultdict(int), defaultdict(int), defaultdict(int), defaultdict(int)
    for x in nums:
        c = (1 + cI[x - 1]) % MOD; s = (x * c + sI[x - 1]) % MOD
        cI[x] = (cI[x] + c) % MOD; sI[x] = (sI[x] + s) % MOD
        c = (1 + cD[x + 1]) % MOD; s = (x * c + sD[x + 1]) % MOD
        cD[x] = (cD[x] + c) % MOD; sD[x] = (sD[x] + s) % MOD
    return (sum(sI.values()) + sum(sD.values()) - sum(nums)) % MOD

for nums, exp in examples:
    print(dp_value(nums), '| kỳ vọng', exp)

6 | kỳ vọng 6
31 | kỳ vọng 31


### 7.3. Cách 4 — Đóng góp trái × phải (cách của doocs)

Với chiều tăng: `L_i` = số dãy tăng-1 kết thúc tại một vị trí `< i` với giá trị `x − 1`; `R_i` = số dãy tăng-1 bắt đầu tại vị trí `> i` với giá trị `x + 1`. Phần tử `i` nằm trong `L_i + R_i + L_i·R_i` dãy tăng-1 **độ dài ≥ 2** (nối bên trái, bên phải, hoặc cả hai). Tương tự cho chiều giảm (đảo mảng). Cộng `Σ nums` cho các dãy một phần tử.

In [6]:
def contribution(nums):
    def calc(a):
        n = len(a)
        endc = defaultdict(int)                 # số dãy tăng-1 kết thúc bằng giá trị v (trong tiền tố)
        L = [0] * n
        for i, x in enumerate(a):
            L[i] = endc[x - 1]
            endc[x] = (endc[x] + 1 + endc[x - 1]) % MOD
        startc = defaultdict(int)               # số dãy tăng-1 bắt đầu bằng giá trị v (trong hậu tố)
        R = [0] * n
        for i in range(n - 1, -1, -1):
            x = a[i]
            R[i] = startc[x + 1]
            startc[x] = (startc[x] + 1 + startc[x + 1]) % MOD
        return sum((l + r + l * r) % MOD * x for l, r, x in zip(L, R, a)) % MOD
    return (calc(nums) + calc(nums[::-1]) + sum(nums)) % MOD

for nums, exp in examples:
    print(contribution(nums), '| kỳ vọng', exp)

6 | kỳ vọng 6
31 | kỳ vọng 31


Đảo mảng biến dãy giảm-1 (theo thứ tự chỉ số) thành dãy tăng-1 ⇒ dùng lại `calc`.

## 8. TRACE từng bước

Trace cách 3 trên `[2, 3, 2, 1, 3]`.

In [7]:
def trace(nums):
    cI, sI, cD, sD = defaultdict(int), defaultdict(int), defaultdict(int), defaultdict(int)
    for i, x in enumerate(nums):
        c1 = 1 + cI[x - 1]; s1 = x * c1 + sI[x - 1]; cI[x] += c1; sI[x] += s1
        c2 = 1 + cD[x + 1]; s2 = x * c2 + sD[x + 1]; cD[x] += c2; sD[x] += s2
        print(f"i={i} x={x}: tăng (+{c1} dãy, +{s1}), giảm (+{c2} dãy, +{s2})")
    inc, dec = sum(sI.values()), sum(sD.values())
    print(f'INC = {inc}, DEC = {dec}, Σ = {sum(nums)} -> đáp án {inc + dec - sum(nums)} | vét cạn {brute(nums)}')

trace([2, 3, 2, 1, 3])

i=0 x=2: tăng (+1 dãy, +2), giảm (+1 dãy, +2)
i=1 x=3: tăng (+2 dãy, +8), giảm (+1 dãy, +3)
i=2 x=2: tăng (+1 dãy, +2), giảm (+2 dãy, +7)
i=3 x=1: tăng (+1 dãy, +1), giảm (+4 dãy, +13)
i=4 x=3: tăng (+3 dãy, +13), giảm (+1 dãy, +3)
INC = 26, DEC = 28, Σ = 11 -> đáp án 43 | vét cạn 43


## 9. Code nộp LeetCode

(Chữ ký theo doocs — xem ghi chú Premium.)

In [8]:
class Solution:
    def getSum(self, nums: List[int]) -> int:
        MOD = 10 ** 9 + 7
        M = max(nums) + 2
        cI = [0] * (M + 1); sI = [0] * (M + 1)
        cD = [0] * (M + 1); sD = [0] * (M + 1)
        total = 0
        for x in nums:
            c = 1 + cI[x - 1]                       # x >= 1 nên x - 1 >= 0
            s = (x * c + sI[x - 1]) % MOD
            cI[x] = (cI[x] + c) % MOD
            sI[x] = (sI[x] + s) % MOD
            c2 = 1 + cD[x + 1]
            s2 = (x * c2 + sD[x + 1]) % MOD
            cD[x] = (cD[x] + c2) % MOD
            sD[x] = (sD[x] + s2) % MOD
            total += s + s2 - x
        return total % MOD

for nums, exp in examples:
    got = Solution().getSum(nums)
    print(got, 'OK' if got == exp else 'SAI')

6 OK
31 OK


Cộng dồn `s + s2 − x` ngay mỗi bước: `s`, `s2` là tổng các dãy **kết thúc tại vị trí này** (mỗi chiều), trừ `x` vì dãy `[x]` bị tính hai lần.

## 10. Vì sao thuật toán đúng

1. **Phân hoạch:** dãy con liên tiếp độ dài `≥ 2` có bước cố định `+1` hoặc `−1` — thuộc đúng một chiều; độ dài 1 thuộc cả hai ⇒ `INC + DEC − Σ nums`.
2. **DP một chiều (tăng):** dãy tăng-1 kết thúc tại vị trí `i` (giá trị `x`) hoặc là `[x]`, hoặc là dãy tăng-1 kết thúc ở vị trí `< i` với giá trị `x − 1` nối thêm `x` — song ánh. `cntI[v]`, `smI[v]` gom mọi dãy kết thúc bằng giá trị `v` trong tiền tố ⇒ `c`, `s` đúng.
3. **Đóng góp:** dãy tăng-1 chứa `i` (độ dài ≥ 2) được xác định duy nhất bởi phần trước `i` (rỗng hoặc một dãy kết thúc bằng `x − 1`) và phần sau (rỗng hoặc bắt đầu bằng `x + 1`), không đồng thời rỗng ⇒ `(1 + L)(1 + R) − 1 = L + R + LR`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP theo chỉ số | `O(n²)` | `O(n)` |
| 3. DP theo giá trị | `O(n)` | `O(max(nums))` |
| 4. Đóng góp | `O(n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'dp_quadratic': dp_quadratic, 'dp_value': dp_value, 'contribution': contribution,
         'submit': lambda a: Solution().getSum(a)}
edge = [([5], 5), ([2, 1], 6), ([1, 1], 2), ([1, 3], 4), ([3, 4, 3], 24), ([1, 2, 3], 20), ([2, 2, 3], 17)]
for nums, exp in examples + edge:
    assert brute(nums) == exp, (nums, brute(nums))
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums, f(nums))
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [10]:
import random
rng = random.Random(3299)
cnt = 0
for _ in range(600):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 12))]
    exp = brute(nums)
    for name, f in impls.items():
        assert f(nums) == exp, (name, nums)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'mảng ngẫu nhiên.')

Khớp vét cạn trên 600 mảng ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên 1..1e5': [rng.randint(1, 10 ** 5) for _ in range(N)],
    'ngẫu nhiên 1..5': [rng.randint(1, 5) for _ in range(N)],
    'tăng 1..N': list(range(1, N + 1)),
    'răng cưa 1,2,1,2': [1 + i % 2 for i in range(N)],
}
for name, nums in big.items():
    t = time.time(); a = Solution().getSum(nums); t1 = time.time() - t
    t = time.time(); b = contribution(nums); t2 = time.time() - t
    assert a == b
    print(f"{name:18s}: {a:10d} | nộp {t1:.2f}s | đóng góp {t2:.2f}s")

ngẫu nhiên 1..1e5 :  171568775 | nộp 0.07s | đóng góp 0.14s
ngẫu nhiên 1..5   :  487211017 | nộp 0.04s | đóng góp 0.10s


tăng 1..N         :  497683716 | nộp 0.05s | đóng góp 0.13s
răng cưa 1,2,1,2  :  500149951 | nộp 0.04s | đóng góp 0.09s


Kiểm tra độc lập: với `1, 2, …, N`, dãy tăng-1 là các đoạn con liên tiếp, dãy giảm-1 chỉ có độ dài 1 ⇒ đáp án = tổng các đoạn con `= Σ i·(i)·(N − i + 1)` (phần tử giá trị `i` ở vị trí `i − 1`).

In [12]:
for N2 in [1, 7, 1000, 10 ** 5]:
    nums = list(range(1, N2 + 1))
    assert Solution().getSum(nums) == sum(v * v * (N2 - v + 1) for v in nums) % MOD
print('dãy 1..N: khớp công thức tổng các đoạn con.')

dãy 1..N: khớp công thức tổng các đoạn con.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép đổi chiều (nhầm với LC 3351)

In [13]:
def wrong_alternating(nums):
    cnt, sm = defaultdict(int), defaultdict(int)
    for x in nums:
        c = 1 + cnt[x - 1] + cnt[x + 1]          # SAI: trộn hai chiều
        s = x * c + sm[x - 1] + sm[x + 1]
        cnt[x] += c; sm[x] += s
    return sum(sm.values()) % MOD

print('sai :', wrong_alternating([3, 4, 3]), '| đúng:', Solution().getSum([3, 4, 3]))

sai : 34 | đúng: 24


**Vì sao sai:** `[3, 4, 3]` đổi chiều nên **không** liên tiếp; bản sai đếm thêm giá trị 10 của nó.

### Lỗi 2 — Quên trừ dãy một phần tử bị đếm hai lần

In [14]:
def wrong_double_singletons(nums):
    return (one_direction(nums, 1) + one_direction(nums, -1)) % MOD   # SAI

print('sai :', wrong_double_singletons([1, 2]), '| đúng:', Solution().getSum([1, 2]))

sai : 9 | đúng: 6


**Vì sao sai:** `[1]`, `[2]` vừa là dãy tăng vừa là dãy giảm.

### Lỗi 3 — Chỉ tính chiều tăng

In [15]:
def wrong_only_increasing(nums):
    return one_direction(nums, 1) % MOD                               # SAI

print('sai :', wrong_only_increasing([1, 4, 2, 3]), '| đúng:', Solution().getSum([1, 4, 2, 3]))

sai : 24 | đúng: 31


**Vì sao sai:** ví dụ 2 có dãy giảm `[4, 3]` (giá trị 7).

### Lỗi 4 — Chỉ cộng `x` một lần khi nối vào `c` dãy

In [16]:
def wrong_add_once(nums):
    total = 0
    for d in (1, -1):
        cnt, sm = defaultdict(int), defaultdict(int)
        for x in nums:
            c = 1 + cnt[x - d]
            s = x + sm[x - d]                                      # SAI: phải là x * c
            cnt[x] += c; sm[x] += s
        total += sum(sm.values())
    return (total - sum(nums)) % MOD

print('sai :', wrong_add_once([1, 2, 3]), '| đúng:', Solution().getSum([1, 2, 3]))

sai : 10 | đúng: 20


**Vì sao sai:** tại `x = 3` có ba dãy mới `[3]`, `[2,3]`, `[1,2,3]` — mỗi dãy chứa `3`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- So sánh với LC 3351: "|hiệu| = 1 giữa hai phần tử kề" (đổi chiều tự do) vs "hiệu **cố định** ±1" — khác biệt nằm ở việc tách hai DP.
- Khi một đối tượng được đếm ở nhiều nhóm (dãy độ dài 1), dùng bao hàm–loại trừ đơn giản.
- Tổng các tổng ⇒ mang `(số lượng, tổng)` hoặc dùng đóng góp từng phần tử.

## 15. Biến thể của bài

1. Bước cố định `d` bất kỳ (cấp số cộng công sai `±d`).
2. Chỉ đếm **số** dãy liên tiếp.
3. Tổng của dãy dài nhất (thay vì mọi dãy).
4. Cấp số cộng với công sai **tùy ý** (LC 446 Arithmetic Slices II).

In [17]:
# Biến thể 1 và 2
def step_d(nums, d):
    if d == 0:
        cnt, sm = defaultdict(int), defaultdict(int)
        for x in nums:
            c = 1 + cnt[x]; s = x * c + sm[x]; cnt[x] += c; sm[x] += s
        return sum(sm.values())
    return one_direction(nums, d) + one_direction(nums, -d) - sum(nums)

def count_only(nums):
    total = 0
    for d in (1, -1):
        cnt = defaultdict(int)
        for x in nums:
            c = 1 + cnt[x - d]; cnt[x] += c; total += c
    return total - len(nums)

rng = random.Random(15)
for _ in range(300):
    nums = [rng.randint(1, 8) for _ in range(rng.randint(1, 10))]
    d = rng.randint(1, 3)
    exp = 0; exp_cnt = 0
    for mask in range(1, 1 << len(nums)):
        seq = [nums[i] for i in range(len(nums)) if mask >> i & 1]
        diffs = {b - a for a, b in zip(seq, seq[1:])}
        if len(seq) == 1 or diffs in ({d}, {-d}):
            exp += sum(seq)
        exp_cnt += is_consecutive(seq)
    assert step_d(nums, d) == exp
    assert count_only(nums) == exp_cnt
print('Biến thể 1 (bước ±d) và 2 (đếm số dãy) khớp vét cạn.')

Biến thể 1 (bước ±d) và 2 (đếm số dãy) khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi dãy con | `O(2^n n)` | Oracle |
| 2 | DP theo chỉ số | `O(n²)` | Dễ nghĩ |
| 3 | DP theo giá trị hai chiều | `O(n)` | **Nộp bài** |
| 4 | Đóng góp trái × phải | `O(n)` | Cách của doocs |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[7, 7]`.

**Bài 2 (tính tay):** `[3, 2, 1]`.

**Bài 3 (điền TODO):** số dãy con liên tiếp **độ dài đúng 3**.

**Bài 4 (tự viết lại):** với mỗi `i`, số dãy con liên tiếp chứa `i`.

**Bài 5 (mini challenge):** LC 446 — đếm dãy con cấp số cộng (độ dài ≥ 3, công sai tùy ý).

In [18]:
def exactly_three_todo(nums):
    # TODO
    raise NotImplementedError

def containing_todo(nums):
    # TODO
    raise NotImplementedError

def arithmetic_slices_todo(nums):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không có cặp bước ±1. (2) Chỉ hai dãy độ dài 1. (3) `14`.

**Bài 2.** (1) Giảm-1: mọi đoạn con. (2) `3+2+1 + 5+3 + 6`. (3) `20`.

**Bài 3.** (1) DP theo độ dài: `cnt[len][v]`, `len ≤ 3`. (2) Hai chiều. (3) Không trừ gì (độ dài 3 không trùng).

**Bài 4.** (1) `1 + (L + R + LR)` cho mỗi chiều… cẩn thận dãy `[x]` chỉ tính 1 lần. (2) `(1 + L_I)(1 + R_I) + (1 + L_D)(1 + R_D) − 1`. (3) Vét cạn.

**Bài 5.** (1) `dp[i][d]` = số dãy (độ dài ≥ 2) kết thúc tại `i` công sai `d`. (2) `dp[i][d] += dp[j][d] + 1`. (3) Đáp án `Σ dp[j][d]` khi nối (độ dài ≥ 3).

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().getSum([7, 7]) == 14
assert Solution().getSum([3, 2, 1]) == 20 == brute([3, 2, 1])

def exactly_three(nums):
    total = 0
    for d in (1, -1):
        c1, c2 = defaultdict(int), defaultdict(int)       # số dãy độ dài 1, 2 kết thúc bằng v
        for x in nums:
            total += c2[x - d]
            c2[x] += c1[x - d]
            c1[x] += 1
    return total

rng = random.Random(3)
for _ in range(200):
    nums = [rng.randint(1, 5) for _ in range(rng.randint(1, 10))]
    exp = sum(1 for mask in range(1, 1 << len(nums)) if bin(mask).count('1') == 3
              and is_consecutive([nums[i] for i in range(len(nums)) if mask >> i & 1]))
    assert exactly_three(nums) == exp
print('Bài 1: 14 | Bài 2: 20 | Bài 3 khớp vét cạn.')

Bài 1: 14 | Bài 2: 20 | Bài 3 khớp vét cạn.


In [20]:
# Bài 4, 5
def containing(nums):
    def LR(a):
        n = len(a); endc = defaultdict(int); L = [0] * n
        for i, x in enumerate(a):
            L[i] = endc[x - 1]; endc[x] += 1 + endc[x - 1]
        st = defaultdict(int); R = [0] * n
        for i in range(n - 1, -1, -1):
            x = a[i]; R[i] = st[x + 1]; st[x] += 1 + st[x + 1]
        return L, R
    LI, RI = LR(nums)
    LD, RD = LR(nums[::-1]); LD, RD = LD[::-1], RD[::-1]
    return [(1 + a) * (1 + b) + (1 + c) * (1 + d) - 1 for a, b, c, d in zip(LI, RI, LD, RD)]

def arithmetic_slices(nums):
    dp = [defaultdict(int) for _ in nums]
    ans = 0
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]
            ans += dp[j][d]
            dp[i][d] += dp[j][d] + 1
    return ans

rng = random.Random(4)
for _ in range(200):
    nums = [rng.randint(1, 5) for _ in range(rng.randint(1, 9))]
    n = len(nums); exp = [0] * n; exp_ap = 0
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        seq = [nums[i] for i in idx]
        if is_consecutive(seq):
            for i in idx:
                exp[i] += 1
        if len(seq) >= 3 and len({b - a for a, b in zip(seq, seq[1:])}) == 1:
            exp_ap += 1
    assert containing(nums) == exp
    assert arithmetic_slices(nums) == exp_ap
assert arithmetic_slices([2, 4, 6, 8, 10]) == 7
print('Bài 4, 5 khớp vét cạn (LC 446 ví dụ [2,4,6,8,10] -> 7).')

Bài 4, 5 khớp vét cạn (LC 446 ví dụ [2,4,6,8,10] -> 7).


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
for x in nums:
    c = 1 + cI[x-1]; s = x*c + sI[x-1]; cI[x] += c; sI[x] += s     # tăng
    c2 = 1 + cD[x+1]; s2 = x*c2 + sD[x+1]; cD[x] += c2; sD[x] += s2 # giảm
    total += s + s2 - x                                              # [x] đếm 2 lần
```

**Checklist:**

- [ ] Hai chiều **tách riêng** (không trộn `x − 1` và `x + 1`)?
- [ ] Trừ `Σ nums` (dãy một phần tử)?
- [ ] Cộng `x · c`?
- [ ] Cộng dồn `cnt[x] += c` (không gán đè)?
- [ ] Modulo?